In [ ]:
# ============================================================
# Programmatic probe -> HGNC symbol mapping for Model B top-50 panel
# Ingests probes directly from '46gene_panel_stability_STRICT_top50.csv'
# to eliminate hardcoding and maintain pipeline reproducibility.
# ============================================================

if (!requireNamespace("hgu133plus2.db", quietly = TRUE)) {
  install.packages("BiocManager")
  BiocManager::install("hgu133plus2.db", update = FALSE, ask = FALSE)
}
library(hgu133plus2.db)
library(AnnotationDbi)

# 1. Programmatically load target probes from disk artifact
stability_file <- "../results/feature_stability/46gene_panel_stability_STRICT_top50.csv"
stability_df <- read.csv(stability_file, stringsAsFactors = FALSE)

if ("probe_id" %in% colnames(stability_df)) {
  top50_probes <- stability_df$probe_id
} else if ("Probe" %in% colnames(stability_df)) {
  top50_probes <- stability_df$Probe
} else {
  top50_probes <- stability_df[, 1]
}

cat("Successfully loaded", length(top50_probes), "probes from", stability_file, "\n\n")

# 2. Query manufacturer database (hgu133plus2.db)
symbols <- AnnotationDbi::select(
  hgu133plus2.db,
  keys = top50_probes,
  keytype = "PROBEID",
  columns = c("SYMBOL", "GENENAME")
)

# 3. Collapse multiple mapping rows and fix SACK1D for publication/cBioPortal
symbols_clean <- aggregate(SYMBOL ~ PROBEID, data = symbols, FUN = function(x) {
  unique_syms <- unique(x)

  # Standardize SACK1D back to FAM83D for database compatibility
  #(As cBioPortal still uses the legacy gene symbol, FAM83D)
  if ("SACK1D" %in% unique_syms) {
    unique_syms[unique_syms == "SACK1D"] <- "FAM83D"
  }

  paste(unique(unique_syms), collapse = "; ")
})

# 4. Save hand-off artifact for Notebook 04
write.csv(symbols_clean, "../results/model_B_top50_reannotated.csv", row.names = FALSE)
print(symbols_clean)

cat("\nStill unmapped (NA) after hgu133plus2.db lookup:\n")
print(setdiff(top50_probes, symbols_clean$PROBEID))

Successfully loaded 50 probes from 46gene_panel_stability_STRICT_top50.csv 



'select()' returned 1:1 mapping between keys and columns



        PROBEID   SYMBOL
1  1555270_a_at     WFS1
2     200934_at      DEK
3   201381_x_at   CACYBP
4     201388_at    PSMD3
5     201710_at    MYBL2
6     202954_at    UBE2C
7     202991_at   STARD3
8   203380_x_at    SRSF5
9     203418_at    CCNA2
10    203967_at     CDC6
11  203968_s_at     CDC6
12  204092_s_at    AURKA
13  204126_s_at    CDC45
14    205225_at     ESR1
15    206364_at    KIF14
16  210085_s_at    ANXA9
17  210761_s_at     GRB7
18  210930_s_at    ERBB2
19    213611_at     AQP5
20  214700_x_at     RIF1
21    214858_at GPC1-AS1
22  218211_s_at     MLPH
23    218319_at    PELI1
24    218447_at     CMC2
25    218976_at  DNAJC12
26    219455_at   CFAP69
27  219555_s_at    CENPN
28    219867_at    CHODL
29    220054_at    IL23A
30  220651_s_at    MCM10
31  221677_s_at   DONSON
32    221811_at    PGAP3
33    223259_at   ORMDL3
34    223523_at  TMEM108
35    223700_at     MND1
36  224447_s_at    MIEN1
37    224753_at    CDCA5
38    225421_at   PM20D2
39    225687_at   FAM83D
